In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])


0

In [1]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Python executable:", sys.executable)
print("Transformers:", transformers.__version__, transformers.__file__)
print("Accelerate:", accelerate.__version__, accelerate.__file__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Python executable: /usr/bin/python3
Transformers: 4.52.4 /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Accelerate: 1.7.0 /usr/local/lib/python3.12/dist-packages/accelerate/__init__.py
Datasets: 3.6.0
Torch: 2.7.1+cu126


In [9]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)



In [10]:
# =========================================================
# 2. GOOGLE DRIVE + SETUP
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

SEED = 42
K = 5
MODEL_NAME = "roberta-base"

OUTPUT_DIR = "/content/drive/MyDrive/RoBERTa_Hierarchical_ManualKFold"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [11]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaModel,
    RobertaConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    
)

In [12]:
# =========================================================
# 2. GOOGLE DRIVE
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

os.makedirs("/content/drive/MyDrive", exist_ok=True)

LOG_FILE_STEP1 = "/content/drive/MyDrive/RoBERTa_Hierarchical_Step1.csv"
LOG_FILE_STEP2 = "/content/drive/MyDrive/RoBERTa_Hierarchical_Step2.csv"

for LOG_FILE, STEP_NAME in [
    (LOG_FILE_STEP1, "STEP 1"),
    (LOG_FILE_STEP2, "STEP 2")
]:
    with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

      
        writer.writerow(["model", "roberta-base"])
        writer.writerow(["step", STEP_NAME])
        writer.writerow(["learning_rate", 2e-5])
        writer.writerow(["train_batch_size", 16,16])
        writer.writerow(["eval_batch_size", 16,16])
        writer.writerow(["epochs", 3,10])
        writer.writerow([])


# =========================================================
# 3. SEED + DEVICE
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)






Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [13]:
# =========================================================
# 5. LOAD DATASET
# =========================================================
train_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val_data   = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test_data  = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")

train_df = train_data.to_pandas()
val_df   = val_data.to_pandas()
test_df  = test_data.to_pandas()

print("Original sizes:")
print({
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
})

# =========================================================
# 4. LABELS
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
INTENSITY_COLUMNS = [f"{emotion}_intensity" for emotion in EMOTIONS]

LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]


# =========================================================
# 6. PREPARE TWO-STEP DATA
# =========================================================
def prepare_two_step_data(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_intensity"] = df[emotion].astype(int)

    for emotion in EMOTIONS:
        df[emotion] = (df[f"{emotion}_intensity"] > 0).astype(int)

    return df

train_two = prepare_two_step_data(train_df)
val_two   = prepare_two_step_data(val_df)
test_two  = prepare_two_step_data(test_df)

# =========================================================
# 7. TRUE 70/20/10 SPLIT
# =========================================================
full_df = pd.concat([train_two, val_two, test_two], ignore_index=True)
full_df = full_df[["text"] + EMOTIONS + INTENSITY_COLUMNS]
full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n = len(full_df)
train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_split = full_df[:train_end].reset_index(drop=True)
val_split   = full_df[train_end:val_end].reset_index(drop=True)
test_split  = full_df[val_end:].reset_index(drop=True)

print("New split sizes:")
print({
    "train": len(train_split),
    "val": len(val_split),
    "test": len(test_split)
})

print("\nSample data:")
print(train_df.head())

# =========================================================
# 8. STEP 1 DATA (EMOTION PRESENCE)
# =========================================================
train_step1_df = train_split[["text"] + EMOTIONS].copy()
val_step1_df   = val_split[["text"] + EMOTIONS].copy()
test_step1_df  = test_split[["text"] + EMOTIONS].copy()

# =========================================================
# 9. STEP 2 DATA (INTENSITY PREDICTION)
# =========================================================
train_step2_df = train_split[["text"] + INTENSITY_COLUMNS].copy()
val_step2_df   = val_split[["text"] + INTENSITY_COLUMNS].copy()
test_step2_df  = test_split[["text"] + INTENSITY_COLUMNS].copy()

# =========================================================
# 10. CONVERT TO HF DATASETS
# =========================================================
train_step1_ds = Dataset.from_pandas(train_step1_df, preserve_index=False)
val_step1_ds   = Dataset.from_pandas(val_step1_df, preserve_index=False)
test_step1_ds  = Dataset.from_pandas(test_step1_df, preserve_index=False)

train_step2_ds = Dataset.from_pandas(train_step2_df, preserve_index=False)
val_step2_ds   = Dataset.from_pandas(val_step2_df, preserve_index=False)
test_step2_ds  = Dataset.from_pandas(test_step2_df, preserve_index=False)

# =========================================================
# 11. TOKENIZER
# =========================================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_step1_ds = train_step1_ds.map(tokenize_function, batched=True)
val_step1_ds   = val_step1_ds.map(tokenize_function, batched=True)
test_step1_ds  = test_step1_ds.map(tokenize_function, batched=True)

train_step2_ds = train_step2_ds.map(tokenize_function, batched=True)
val_step2_ds   = val_step2_ds.map(tokenize_function, batched=True)
test_step2_ds  = test_step2_ds.map(tokenize_function, batched=True)

# =========================================================
# 12. ADD LABELS
# =========================================================
def add_step1_labels(example):
    example["labels"] = [float(example[emotion]) for emotion in EMOTIONS]
    return example

def add_step2_labels(example):
    example["labels"] = [int(example[col]) for col in INTENSITY_COLUMNS]
    return example

train_step1_ds = train_step1_ds.map(add_step1_labels)
val_step1_ds   = val_step1_ds.map(add_step1_labels)
test_step1_ds  = test_step1_ds.map(add_step1_labels)

train_step2_ds = train_step2_ds.map(add_step2_labels)
val_step2_ds   = val_step2_ds.map(add_step2_labels)
test_step2_ds  = test_step2_ds.map(add_step2_labels)

# =========================================================
# 13. FORMAT
# =========================================================
train_step1_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_step1_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_step1_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_step2_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_step2_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_step2_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])



Original sizes:
{'train': 2763, 'val': 115, 'test': 2765}
New split sizes:
{'train': 3950, 'val': 1128, 'test': 565}

Sample data:
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         0  
4      0      NaN     3    0        1         2  


Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# 15. STEP 1 METRICS
# =========================================================
def compute_metrics_step1(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": float(np.mean(pearsons))
    }


# =========================================================
# 18. STEP 1 CALLBACK
# =========================================================
class SaveMetricsCallbackStep1(TrainerCallback):
    def __init__(self, file_path, test_dataset):
        self.file_path = file_path
        self.test_dataset = test_dataset

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

        self.epoch_list = []
        self.train_loss_list = []

        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []

        self.test_loss_list = []
        self.test_f1_macro_list = []
        self.test_f1_micro_list = []
        self.test_pearson_mean_list = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval or metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))
        test_f1_macro = float(test_results.get("test_f1_macro", 0.0))
        test_f1_micro = float(test_results.get("test_f1_micro", 0.0))
        test_pearson_mean = float(test_results.get("test_pearson_mean", 0.0))

        self.epoch_list.append(epoch)
        self.train_loss_list.append(train_loss)

        self.val_loss_list.append(val_loss)
        self.val_f1_macro_list.append(val_f1_macro)
        self.val_f1_micro_list.append(val_f1_micro)
        self.val_pearson_mean_list.append(val_pearson_mean)

        self.test_loss_list.append(test_loss)
        self.test_f1_macro_list.append(test_f1_macro)
        self.test_f1_micro_list.append(test_f1_micro)
        self.test_pearson_mean_list.append(test_pearson_mean)

        pred = self.trainer_ref.predict(self.test_dataset)

        probs = 1 / (1 + np.exp(-pred.predictions))
        pred_labels = (probs >= 0.5).astype(int)
        true_labels = pred.label_ids

        report_dict = classification_report(
            true_labels,
            pred_labels,
            target_names=EMOTIONS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(self.epoch_list)):
                writer.writerow([
                    self.epoch_list[i],
                    self.train_loss_list[i],
                    self.val_loss_list[i],
                    self.test_loss_list[i],
                    self.val_f1_macro_list[i],
                    self.val_f1_micro_list[i],
                    self.test_f1_macro_list[i],
                    self.test_f1_micro_list[i],
                    self.val_pearson_mean_list[i],
                    self.test_pearson_mean_list[i]
                ])

            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow(["class", "precision", "recall", "f1_score", "support"])

            for class_name in EMOTIONS:
                row = report_dict.get(class_name, {})
                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(f"\nStep 1 epoch {epoch} results saved.")

        self._inside_eval = False


# =========================================================
# 19. STEP 1 MODEL
# =========================================================
model_step1 = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels= len(EMOTIONS),
    problem_type="multi_label_classification"
)


# =========================================================
# 20. STEP 1 TRAINING ARGS
# =========================================================
training_args_step1 = TrainingArguments(
    output_dir="/content/roberta_step1_output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 21. STEP 1 TRAINER
# =========================================================
callback_step1 = SaveMetricsCallbackStep1(
    file_path=LOG_FILE_STEP1,
    test_dataset=test_step1_ds
)

trainer_step1 = Trainer(
    model=model_step1,
    args=training_args_step1,
    train_dataset=train_step1_ds,
    eval_dataset=val_step1_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_step1,
    callbacks=[callback_step1]
)

callback_step1.trainer_ref = trainer_step1


# =========================================================
# 22. TRAIN STEP 1
# =========================================================
print("\nStarting Step 1: Emotion Detection")

start1 = time.time()
trainer_step1.train()
end1 = time.time()

print(f"Step 1 training time: {end1 - start1:.1f} seconds")
print("Step 1 epochwise result file saved at:", LOG_FILE_STEP1)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting Step 1: Emotion Detection


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Pearson Mean
1,0.487100,0.381627,0.579098,0.692405,0.635398
2,0.346800,0.339558,0.715400,0.753805,0.678689
3,0.275800,0.340599,0.704585,0.746022,0.683279
4,0.231000,0.329272,0.734529,0.766084,0.693735



Step 1 epoch 1 results saved.

Step 1 epoch 2 results saved.

Step 1 epoch 3 results saved.

Step 1 epoch 4 results saved.
Step 1 training time: 152.5 seconds
Step 1 epochwise result file saved at: /content/drive/MyDrive/RoBERTa_Hierarchical_Step1.csv


In [49]:
# =========================================================
# 16. STEP 2 MODEL
# =========================================================
class RobertaStep2IntensityModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained(
            "roberta-base",
        )

        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            hidden_size,
            len(EMOTIONS) * 4
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)

        logits = logits.view(
            -1,
            len(EMOTIONS),
            4
        )

        loss = None

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()

            loss = loss_fct(
                logits.view(-1, 4),
                labels.view(-1)
            )

        return {
            "loss": loss,
            "logits": logits
        }


# =========================================================
# 17. STEP 2 METRICS
# =========================================================
def compute_metrics_step2(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    true_flat = labels.reshape(-1)
    pred_flat = preds.reshape(-1)

    f1_macro = f1_score(
        true_flat,
        pred_flat,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        true_flat,
        pred_flat,
        average="micro",
        zero_division=0
    )

    if np.std(true_flat) == 0 or np.std(pred_flat) == 0:
        pearson_mean = 0.0
    else:
        pearson_mean, _ = pearsonr(true_flat, pred_flat)
        pearson_mean = 0.0 if np.isnan(pearson_mean) else float(pearson_mean)

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }
# =========================================================
# 23. STEP 2 CALLBACK
# =========================================================
class SaveMetricsCallbackStep2(TrainerCallback):
    def __init__(
        self,
        file_path,
        step1_trainer,
        test_step1_dataset,
        test_step2_dataset
    ):
        self.file_path = file_path
        self.step1_trainer = step1_trainer
        self.test_step1_dataset = test_step1_dataset
        self.test_step2_dataset = test_step2_dataset

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

        self.epoch_list = []
        self.train_loss_list = []

        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []

        self.test_loss_list = []
        self.test_f1_macro_list = []
        self.test_f1_micro_list = []
        self.test_pearson_mean_list = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def make_final_hierarchical_predictions(self):
        pred_step1 = self.step1_trainer.predict(self.test_step1_dataset)

        probs_step1 = 1 / (1 + np.exp(-pred_step1.predictions))
        pred_emotions = (probs_step1 >= 0.5).astype(int)

        pred_step2 = self.trainer_ref.predict(self.test_step2_dataset)

        logits_step2 = pred_step2.predictions
        true_intensities = pred_step2.label_ids

        probs_step2 = torch.softmax(
            torch.tensor(logits_step2),
            dim=-1
        ).numpy()

        pred_intensities = np.argmax(probs_step2, axis=-1)

        final_pred_intensities = pred_intensities * pred_emotions

        true_binary_15 = np.zeros(
            (true_intensities.shape[0], len(LABELS)),
            dtype=int
        )

        pred_binary_15 = np.zeros(
            (true_intensities.shape[0], len(LABELS)),
            dtype=int
        )

        prob_binary_15 = np.zeros(
            (true_intensities.shape[0], len(LABELS)),
            dtype=float
        )

        for emotion_idx, emotion in enumerate(EMOTIONS):
            for level in [1, 2, 3]:
                col_idx = emotion_idx * 3 + (level - 1)

                true_binary_15[:, col_idx] = (
                    true_intensities[:, emotion_idx] == level
                ).astype(int)

                pred_binary_15[:, col_idx] = (
                    final_pred_intensities[:, emotion_idx] == level
                ).astype(int)

                prob_binary_15[:, col_idx] = (
                    probs_step1[:, emotion_idx]
                    * probs_step2[:, emotion_idx, level]
                )

        return true_binary_15, pred_binary_15, prob_binary_15

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval or metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_step2_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))

        true_binary_15, pred_binary_15, prob_binary_15 = (
            self.make_final_hierarchical_predictions()
        )

        test_f1_macro = f1_score(
            true_binary_15,
            pred_binary_15,
            average="macro",
            zero_division=0
        )

        test_f1_micro = f1_score(
            true_binary_15,
            pred_binary_15,
            average="micro",
            zero_division=0
        )

        pearsons = []

        for i in range(true_binary_15.shape[1]):
            if np.std(true_binary_15[:, i]) == 0 or np.std(prob_binary_15[:, i]) == 0:
                pearsons.append(0.0)
            else:
                p, _ = pearsonr(true_binary_15[:, i], prob_binary_15[:, i])
                pearsons.append(0.0 if np.isnan(p) else float(p))

        test_pearson_mean = float(np.mean(pearsons))

        self.epoch_list.append(epoch)
        self.train_loss_list.append(train_loss)

        self.val_loss_list.append(val_loss)
        self.val_f1_macro_list.append(val_f1_macro)
        self.val_f1_micro_list.append(val_f1_micro)
        self.val_pearson_mean_list.append(val_pearson_mean)

        self.test_loss_list.append(test_loss)
        self.test_f1_macro_list.append(test_f1_macro)
        self.test_f1_micro_list.append(test_f1_micro)
        self.test_pearson_mean_list.append(test_pearson_mean)

        report_dict = classification_report(
            true_binary_15,
            pred_binary_15,
            target_names=LABELS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(self.epoch_list)):
                writer.writerow([
                    self.epoch_list[i],
                    self.train_loss_list[i],
                    self.val_loss_list[i],
                    self.test_loss_list[i],
                    self.val_f1_macro_list[i],
                    self.val_f1_micro_list[i],
                    self.test_f1_macro_list[i],
                    self.test_f1_micro_list[i],
                    self.val_pearson_mean_list[i],
                    self.test_pearson_mean_list[i]
                ])

            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow(["class", "precision", "recall", "f1_score", "support"])

            for class_name in LABELS:
                row = report_dict.get(class_name, {})
                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(f"\nStep 2 epoch {epoch} final hierarchical results saved.")

        self._inside_eval = False


# =========================================================
# 24. STEP 2 MODEL
# =========================================================
model_step2 = RobertaStep2IntensityModel()


# =========================================================
# 25. STEP 2 TRAINING ARGS
# =========================================================
training_args_step2 = TrainingArguments(
    output_dir="/content/roberta_step2_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 26. STEP 2 TRAINER
# =========================================================
callback_step2 = SaveMetricsCallbackStep2(
    file_path=LOG_FILE_STEP2,
    step1_trainer=trainer_step1,
    test_step1_dataset=test_step1_ds,
    test_step2_dataset=test_step2_ds
)

trainer_step2 = Trainer(
    model=model_step2,
    args=training_args_step2,
    train_dataset=train_step2_ds,
    eval_dataset=val_step2_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_step2,
    callbacks=[callback_step2]
)

callback_step2.trainer_ref = trainer_step2


# =========================================================
# 27. TRAIN STEP 2
# =========================================================
print("\nStarting Step 2: Intensity Classification")

start2 = time.time()
trainer_step2.train()
end2 = time.time()

print(f"Step 2 training time: {end2 - start2:.1f} seconds")
print("Step 2 epochwise result file saved at:", LOG_FILE_STEP2)




Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting Step 2: Intensity Classification


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Pearson Mean
1,0.736937,0.615251,0.514595,0.747518,0.667907
2,0.536230,0.584117,0.567685,0.753014,0.731362
3,0.421026,0.592656,0.579726,0.775355,0.746153
4,0.334332,0.611304,0.575850,0.772518,0.756840
5,0.265707,0.641144,0.600397,0.778723,0.762581
6,0.209860,0.686652,0.611765,0.772695,0.768412
7,0.166074,0.710115,0.607457,0.779078,0.766082
8,0.129950,0.760073,0.613019,0.775355,0.761365
9,0.105260,0.772892,0.617395,0.782447,0.766401
10,0.088039,0.784880,0.616904,0.781738,0.770187



Step 2 epoch 1 final hierarchical results saved.


...


Step 2 epoch 6 final hierarchical results saved.



Step 2 epoch 7 final hierarchical results saved.



Step 2 epoch 8 final hierarchical results saved.



Step 2 epoch 9 final hierarchical results saved.



Step 2 epoch 10 final hierarchical results saved.
Step 2 training time: 586.7 seconds
Step 2 epochwise result file saved at: /content/drive/MyDrive/RoBERTa_Hierarchical_Step2.csv
